# Rec Systems V
# Matrix Factorization

This workbook is part of a series, listed below:
- <a href="https://github.com/pw598/Articles/blob/main/Rec%20Systems%20I%20-%20Baseline%20Methods.ipynb">Rec Systems I - Baseline Methods</a>
- <a href="https://github.com/pw598/Articles/blob/main/Rec%20Systems%20II%20-%20Content%20Based.ipynb">Rec Systems II - Content Based</a>
- <a href="https://github.com/pw598/Articles/blob/main/Rec%20Systems%20III%20-%20User-Based%20Collaborative%20Filtering.ipynb">Rec Systems III - User-Based Collaborative Filtering</a>
- <a href="https://github.com/pw598/Articles/blob/main/Rec%20Systems%20IV%20-%20Item-Based%20Collaborative%20Filtering.ipynb">Rec Systems IV - Item-Based Collaborative Filtering</a>
- <a href="https://github.com/pw598/Articles/blob/main/Rec%20Systems%20V%20-%20Matrix%20Factorization.ipynb">Rec Systems V - Matrix Factorization</a>

Previous workbooks looked at baseline methods, content-based methods, and collaborative-filtering methods. This workbook will look at the dimension-reduction methods of matrix factorization and SVD, implemented via the scikit-surprise package.

# Matrix Factorization

In matrix factorization, we start with an $N x M$ matrix with $N$ users and $M$ movies, which is sparse because most users haven't rated most items. We want to approximate it as the product of two smaller matrices.

$$\hat{R} = WU^T$$

where $W$ is an N-row 'users' matrix, and $U$ is an M-row 'movies' matrix. $W$ is $N x K$, and $U^T$ is $K x M$.

The result of multiplying all of $W$ by all of $U^T$ is the original matrix $R$, which may be too large to fit in memory. We want to reduce $W$ and $U$ to being as skinny as possible while also being informative. Doing so will compress the information.

If you only want the prediction for user $i$ and movie $j$ (i.e., $\hat{R}_{ij}$), it is equal to $w_i^T u_j$, the dot product of two vectors of size $K$, with $K$ usually between $10$ and $50$.

### Interpretation

Suppose each of $K$ elements in $w_i$ and $u_j$ represent features - e.g., distinct genres. $w_i^T u_j$ is proportional to cosine similarity, so it informs us how well user $i$'s preferences correlate to movie $j$'s attributes.

$$w_i^T u_j = ||w_i|| ~||u_j|| ~cos \theta \propto sim(i,j)$$

You cannot choose what genres, etc., the features represent. The features are latent, and $K$ is the latent dimensionality.

# Import Libraries and Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surprise import Dataset
from surprise import Reader
from surprise import NMF
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise.model_selection import cross_validate
from surprise.model_selection import GridSearchCV

In [2]:
ratings_data = pd.read_csv("data/ratings_mf.csv")
movies_data = pd.read_csv("data/movies_mf.csv")

In [3]:
min_rating = ratings_data.rating.min()
max_rating = ratings_data.rating.max()
 
reader = Reader(rating_scale=(min_rating, max_rating))
data = Dataset.load_from_df(ratings_data[['userId', 'movieId', 'rating']], reader)

Results can be cross validated.

In [4]:
nmf = NMF(n_epochs=10)
results = cross_validate(nmf, data, measures=['RMSE', 'MAE'], cv=10, verbose=True)

Evaluating RMSE, MAE of algorithm NMF on 10 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Fold 6  Fold 7  Fold 8  Fold 9  Fold 10 Mean    Std     
RMSE (testset)    0.9442  0.9454  0.9387  0.9486  0.9354  0.9403  0.9264  0.9277  0.9434  0.9489  0.9399  0.0076  
MAE (testset)     0.7113  0.7157  0.7105  0.7179  0.7097  0.7161  0.6975  0.7060  0.7146  0.7187  0.7118  0.0061  
Fit time          0.22    0.29    0.23    0.21    0.25    0.23    0.21    0.31    0.30    0.26    0.25    0.04    
Test time         0.07    0.02    0.02    0.03    0.02    0.02    0.02    0.02    0.07    0.02    0.03    0.02    


In [5]:
print("Average MAE: ", np.average(results["test_mae"]))
print("Average RMSE: ", np.average(results["test_rmse"]))

Average MAE:  0.7117994533322187
Average RMSE:  0.9398804527574125


### Grid Search

Parameters such as the number of latent factors can be searched for optimality.

In [6]:
param_grid = {
  'n_factors': [10,15,20,25,30],
  'n_epochs': [15,20,25,30,35,40,45]
}
 
gs = GridSearchCV(NMF, param_grid, measures=['rmse', 'mae'], cv=10)
gs.fit(data)
 
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.9145232020764602
{'n_factors': 15, 'n_epochs': 30}


### Perform a Query

In [7]:
best_factor = gs.best_params['rmse']['n_factors']
best_epoch = gs.best_params['rmse']['n_epochs']

trainset, testset = train_test_split(data, test_size=.20)
nmf = NMF(n_factors=best_factor, n_epochs=best_epoch)
nmf.fit(trainset)

In [8]:
def generate_recommendation(model, user_id, ratings_df, movies_df, n_items):
    
    movie_ids = ratings_df["movieId"].unique()
    movie_ids_user = ratings_df.loc[ratings_df["userId"] == user_id, "movieId"]
    movie_ids_to_pred = np.setdiff1d(movie_ids, movie_ids_user)

    test_set = [[user_id, movie_id, 4] for movie_id in movie_ids_to_pred]
    predictions = model.test(test_set)
    pred_ratings = np.array([pred.est for pred in predictions])
    print("Top {0} item recommendations for user {1}:".format(n_items, user_id))

    index_max = (-pred_ratings).argsort()[:n_items]
    for i in index_max:
        movie_id = movie_ids_to_pred[i]
        print(movies_df[movies_df["movieId"]==movie_id]["title"].values[0], pred_ratings[i])

userID = 20
n_items = 15
generate_recommendation(nmf,userID,ratings_data,movies_data,n_items)

Top 15 item recommendations for user 20:
My Man Godfrey (1936) 5.0
Sherlock Holmes and Dr. Watson: Acquaintance (1979) 5.0
Fugitives (1986) 5.0
There Will Be Blood (2007) 5.0
History of Future Folk, The (2012) 5.0
World of Tomorrow (2015) 5.0
Mystery of the Third Planet, The (Tayna tretey planety) (1981) 5.0
Louis C.K.: Oh My God (2013) 5.0
Black Tar Heroin: The Dark End of the Street (2000) 5.0
Beautiful People (1999) 5.0
20 Feet from Stardom (Twenty Feet from Stardom) (2013) 5.0
Adventures Of Sherlock Holmes And Dr. Watson: The Twentieth Century Approaches (1986) 5.0
Man Bites Dog (C'est arrivé près de chez vous) (1992) 5.0
Visitor, The (2007) 5.0
Party Girl (1995) 5.0


# Singular Value Decomposition (SVD)

SVD tells us that a matrix $X$ can be decomposed into 3 separate matrices multiplied together.

$$X = USV^T$$

where $U$ is $N x K$, $S$ is $K x K$, and $V^T$ is $M x K$. The SVD is exactly equal to $X$ if $K=M$ (assuming $X$ is full-rank and $N > M$). But if we shrink $U$, $S$, and $V$ to being $N x K$, $K x K$, and $K x M$, we get a truncated approximation. We say that $U$, $S$, and $V$ have been truncated, and that $\hat{X}$ is the best $K$-rank approximation of $X$. As with matrix factorization, training is usually done through iterative methods like gradient descent.

In [9]:
svd = SVD(n_epochs=10)
results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=10, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 10 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Fold 6  Fold 7  Fold 8  Fold 9  Fold 10 Mean    Std     
RMSE (testset)    0.8799  0.8860  0.8703  0.8790  0.8654  0.8858  0.8777  0.8719  0.8817  0.8829  0.8781  0.0065  
MAE (testset)     0.6733  0.6810  0.6686  0.6787  0.6704  0.6832  0.6725  0.6717  0.6815  0.6790  0.6760  0.0050  
Fit time          0.30    0.28    0.28    0.31    0.29    0.29    0.30    0.30    0.30    0.30    0.29    0.01    
Test time         0.03    0.03    0.08    0.03    0.03    0.03    0.03    0.03    0.03    0.03    0.03    0.01    


In [10]:
print("Average MAE: ", np.average(results["test_mae"]))
print("Average RMSE: ", np.average(results["test_rmse"]))

Average MAE:  0.6760019250584162
Average RMSE:  0.8780571128214222


### Grid Search

In [11]:
param_grid = {
  'n_factors': [10,15,20,25,30],
  'n_epochs': [15,20,25,30,35,40,45]
}
 
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=10)
gs.fit(data)
 
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8630392388419074
{'n_factors': 10, 'n_epochs': 30}


### Perform a Query

In [12]:
best_factor = gs.best_params['rmse']['n_factors']
best_epoch = gs.best_params['rmse']['n_epochs']

trainset, testset = train_test_split(data, test_size=.20)
svd = SVD(n_factors=best_factor, n_epochs=best_epoch)
svd.fit(trainset)

In [13]:
userID = 20
n_items = 15
generate_recommendation(svd,userID,ratings_data,movies_data,n_items)

Top 15 item recommendations for user 20:
Hours, The (2002) 4.853024641985607
Apocalypse Now (1979) 4.822668116446977
Rear Window (1954) 4.786111269204377
Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001) 4.763910079159124
American Beauty (1999) 4.7066216338155975
Sting, The (1973) 4.67795385688666
Philadelphia Story, The (1940) 4.676533668653466
Cool Hand Luke (1967) 4.6755706951760825
Touch of Evil (1958) 4.67181495120066
Notorious (1946) 4.661276753216377
Shawshank Redemption, The (1994) 4.658321997935041
Hedwig and the Angry Inch (2000) 4.650353571973382
Shaun of the Dead (2004) 4.644818785821565
Wallace & Gromit: A Close Shave (1995) 4.641891350828916
Grand Day Out with Wallace and Gromit, A (1989) 4.638822703343022
